# 🎙️ F5-TTS Türkçe — 10 Saniyede Sıfır Örnekli (Zero-Shot) Ses Klonlama

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beydah/ses-klonlama-turkce-f5-tts/blob/main/Ses-Klonlama_Turkce_F5-TTS.ipynb)
[![Website](https://img.shields.io/badge/Website-beydahsaglam.com-blue?style=flat&logo=googlechrome)](https://beydahsaglam.com)
[![LinkedIn](https://img.shields.io/badge/LinkedIn-beydah-0A66C2?style=flat&logo=linkedin)](https://linkedin.com/in/beydah)
[![GitHub](https://img.shields.io/badge/GitHub-beydah-181717?style=flat&logo=github)](https://github.com/beydah)

Bu Colab çalışma alanı, **F5-TTS (Flow Matching)** mimarisi ve topluluk tarafından eğitilen Türkçe ağırlıklar (`marduk-ra/F5-TTS-Turkish`) ile **yalnızca 10 saniyelik tek bir ses kaydınızdan** kendi sesinizi birebir klonlamanızı sağlar.

### 🚀 Neden F5-TTS ve Akış Eşleme (Flow Matching)?
- **Eğitim Gerektirmez (Zero-Shot):** Saatlerce stüdyo kaydına, günlerce GPU kiralamaya veya LoRA/SFT eğitimine ihtiyaç duymaz.
- **Doğal ve Akıcı:** Eski nesil robotik seslerin veya difüzyondaki fısıltıların aksine, ses tellerinizin titreşimini ve konuşma temponuzu anında yakalar.
- **5 Basit Adım:** Sesi hazırla, yükle, temizle, metni ver ve klonlanmış stüdyo kaydını indir!

In [ ]:
#@title ⚙️ 0. Adım: GPU Kontrolü ve Gerekli Kütüphanelerin Kurulumu
#@markdown T4 GPU kontrol edilir, F5-TTS, Vocos ve ses işleme araçları Colab ortamına kurulur (~40 saniye):

import torch

# 1. GPU Kontrolü (Fail-Fast: CPU'da saatlerce beklemenizi önler)
if not torch.cuda.is_available():
    print("\n" + "!" * 72)
    print("⚠️  DİKKAT: GPU (CUDA) AKTİF DEĞİL! Şu anda CPU ortamındasınız.")
    print("    F5-TTS modeli CPU üzerinde cümle başına ~8-10 dakika sürer.")
    print("    GPU (T4) üzerinde ise cümle başına yalnızca 2-3 saniye sürer!")
    print("\n👉  Hemen GPU'ya geçmek için:")
    print("    1. Yukarıdaki Colab menüsünden: 'Çalışma Zamanı' (Runtime)")
    print("    2. 'Çalışma zamanı türünü değiştir' (Change runtime type)")
    print("    3. Donanım Hızlandırıcı (Hardware accelerator): 'T4 GPU' seçin")
    print("    4. 'Kaydet' butonuna basıp bu hücreyi tekrar çalıştırın.")
    print("!" * 72 + "\n")
    raise SystemExit("Lütfen T4 GPU'ya geçip bu hücreyi tekrar çalıştırın.")

print(f"🚀 GPU Aktif ve Hazır: {torch.cuda.get_device_name(0)} (Cihaz: CUDA)")

import subprocess

print("⏳ Gerekli paketler yükleniyor (f5-tts, pydub, soundfile, scipy)...")
subprocess.run(["pip", "install", "-q", "f5-tts", "pydub", "soundfile", "scipy"], check=True)

import transformers
transformers.pipeline = lambda *args, **kwargs: None

print("✅ Kurulum tamamlandı! Şimdi 1. Adıma geçebilirsiniz.\n")


---
## 🎯 1. Adım: Referans Sesinizi Hazırlayın

Model, ses tonunuzu ve diksiyonunuzu **10 saniyelik** tek bir referans ses kaydından kopyalayacaktır.

### 🎙️ Okunacak 29 Harfli Altın Cümle:
> **"Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir."**

*(Bu cümle Türk alfabesindeki 29 harfin tamamını barındırır; model sesinizin tüm harf tınılarını tek seferde öğrenir.)*

### 💡 Kaliteli Ses İpuçları:
1. **Süre:** Yaklaşık 8 ile 12 saniye arasında okuyun.
2. **Mikrofon:** Mikrofonu ağzınızın tam karşısına değil, hafifçe yanak hizanıza (15-20 cm mesafede) tutun.
3. **Ton:** Fısıldamayın veya yapay bir ciddiyet takınmayın; sanki bir dostunuza heyecanla bir fikir anlatıyormuş gibi enerjik, net ve tok konuşun.

In [ ]:
#@title 📤 2. Adım: Referans Sesinizi Yükleyin
#@markdown Çalıştırın ve açılan butondan kaydettiğiniz ses dosyasını seçin (.wav, .mp3, .ogg, .m4a):

import os
from google.colab import files

print("🎙️ Lütfen 8-12 saniyelik referans sesinizi seçin:")
yuklenen = files.upload()

if not yuklenen:
    raise ValueError("⚠️ Lütfen bir ses dosyası yükleyin!")

ham_ses_yolu = list(yuklenen.keys())[0]
print(f"✅ Ses dosyası yüklendi: {ham_ses_yolu}")

In [ ]:
#@title 🎛️ 3. Adım: Referans Sesi Hazırlayın (İsteğe Bağlı İyileştirme)
#@markdown Zaten temiz veya stüdyo ses kaydınız varsa kutucuğu **İŞARETSİZ (False)** bırakın.

#@markdown ---
#@markdown 🎛️ **İsteğe Bağlı Ses Filtreleme:**
otomatik_iyilestirme_uygula = False #@param {type:"boolean"}
#@markdown *(⚠️ Sadece dip uğultusu veya belirgin gürültü içeren ham kayıtlarda True yapın. Önceden temizlenmiş seste False önerilir).*

import soundfile as sf
import numpy as np
from scipy import signal
from pydub import AudioSegment
from IPython.display import Audio, display

# Her ses formatını standart 48kHz mono WAV formatına dönüştür
seg = AudioSegment.from_file(ham_ses_yolu)
seg = seg.set_channels(1).set_frame_rate(48000)
temiz_referans_yolu = "temiz_referans.wav"
temiz_ses_yolu = temiz_referans_yolu

if otomatik_iyilestirme_uygula:
    print("🎛️ Opsiyonel filtreler uygulanıyor (75Hz dip filtresi, sessizlik kırpma ve normalizasyon)...")
    temp_wav = "temp_raw.wav"
    seg.export(temp_wav, format="wav")
    data, sr = sf.read(temp_wav)
    if len(data.shape) > 1:
        data = data.mean(axis=1)

    # 1. 75 Hz High-pass Butterworth Filtresi
    sos = signal.butter(4, 75, 'hp', fs=sr, output='sos')
    filtered = signal.sosfilt(sos, data)

    # 2. Akıllı Sessizlik Tespiti ve Kırpma (VAD)
    frame_len = int(sr * 0.03)
    energy = np.array([np.sum(filtered[i:i+frame_len]**2) for i in range(0, len(filtered)-frame_len, frame_len)])
    thresh = np.percentile(energy, 65) * 0.08
    active_frames = np.where(energy > thresh)[0]

    if len(active_frames) > 0:
        start_idx = max(0, active_frames[0] * frame_len - int(sr * 0.08))
        end_idx = min(len(filtered), (active_frames[-1] + 1) * frame_len + int(sr * 0.12))
        trimmed = filtered[start_idx:end_idx]
    else:
        trimmed = filtered

    # 3. Yumuşak Kenar Geçişleri
    fade_in = int(sr * 0.015)
    fade_out = int(sr * 0.025)
    if len(trimmed) > (fade_in + fade_out):
        trimmed[:fade_in] *= np.linspace(0, 1, fade_in)
        trimmed[-fade_out:] *= np.linspace(1, 0, fade_out)

    # 4. Normalizasyon (-1.0 dBFS)
    peak = np.max(np.abs(trimmed))
    if peak > 0:
        trimmed = trimmed / peak * 0.89

    sf.write(temiz_referans_yolu, trimmed, sr)
    print(f"✨ Ses filtrelendi! Süre: {len(trimmed)/sr:.2f} saniye")
    print("🎧 Filtrelenen referans sesi dinleyin:")
    display(Audio(temiz_referans_yolu, rate=sr))
else:
    seg.export(temiz_referans_yolu, format="wav")
    data, sr = sf.read(temiz_referans_yolu)
    print(f"ℹ️ Otomatik iyileştirme kapalı: Orijinal ses doğrudan korundu (önerilen, {len(data)/sr:.1f} sn).")
    display(Audio(temiz_referans_yolu, rate=sr))


---
## 🧠 4. Adım: Metni Girin ve Sesi Klonlayın

Aşağıdaki hücrede **hedef_metin** alanına seslendirmek istediğiniz metni doğrudan yapıştırın.

* **Sıfır Halüsinasyon Cümle Mimarisi:** Metin otomatik olarak bağımsız cümlelere bölünür; her cümlenin arasına **250 milisaniyelik doğal insan nefesi** eklenir.
* **Fonetik Sözlük:** React, HTML, JavaScript gibi yabancı teknik terimler seslendirme öncesinde otomatik olarak pürüzsüz Türkçe okunuşlarına (`Riyekt`, `He-Te-Me-Le` vb.) dönüştürülür.

In [ ]:
#@title 🚀 4. Adım: F5-TTS ile Sesi Klonlayın
#@markdown Seslendirme ve model ayarlarını belirleyin:

referans_metni = "Çağdaş projelerde karmaşık sistemleri çözen; özgün, güçlü ve her zaman şeffaf bileşenleri, amaca uygun ve ciddi bir vizyonla tasarlamak gerekir." #@param {type:"string"}
konusma_hizi = 1.0 #@param {type:"slider", min:0.8, max:1.3, step:0.02}
sadakat_cfg = 2.2 #@param {type:"slider", min:1.5, max:3.0, step:0.1}
kalite_adim_sayisi = 32 #@param {type:"slider", min:16, max:32, step:4}

#@markdown ---
#@markdown 📝 **Klonlanacak Hedef Metin:** (İstediğiniz uzunlukta metni aşağıdaki tırnakların içine yapıştırın)
hedef_metin = """
Bir modelin cevabını ekranda göstermek ile modelin arayüz kodunu çalıştırmasına izin vermek farklı kararlardır. Kişisel bir uygulamada ilk deneme için küçük, önceden tanımlanmış bileşenler ve doğrulanmış veri daha denetlenebilir bir başlangıç sunar. Bu yazı üretimde ölçülmüş bir performans sonucu değil, bir uygulama tasarımıdır.

Örneğin model yalnızca özet, kontrol listesi veya kaynak kartı türlerinden birini önerebilsin. Her türün zorunlu alanlarını, metin uzunluklarını ve izin verilen bağlantı biçimlerini belirleyin. Bilinmeyen türü ya da geçersiz adresi render etmek yerine reddedin. Modelin verdiği bileşen veya paket adını dinamik olarak çalıştırmayın.

React metin içeriğini normal şekilde render etmeyi destekler. Ham HTML ekleme özelliği ise güvenilmeyen veri için ayrıca risk taşır. Veriyi doğrulamak tek başına her HTML kullanımını güvenli yapmaz; ham HTML gerekmiyorsa bu yolu hiç açmamak daha anlaşılırdır.

Yüklenirken hangi bölümün hazır olmadığını gösterin. Suspense, destekleyen veri kaynağı veya tembel yüklenen bileşen için fallback sunabilir; sıradan bir effect içindeki isteği kendiliğinden yönetmez. Transition kullanmak da bütün asenkron işi otomatik olarak güvenli veya hızlı yapmaz. Kontrollü metin girişinin güncellemesini transition içine taşımak yerine yazma hissini koruyun.

Geçerli bir kart, eksik zorunlu alan, aşırı uzun metin ve izin verilmeyen bağlantıyla deneyin. Ekran okuyucu için bekleme ve hata mesajlarını kontrol edin. Bir kullanıcı işlemi veri değiştiriyorsa onayı ayrıca alın; model çıktısını yetki olarak kullanmayın.
"""

import os
import re
import time
import torch
import soundfile as sf
import numpy as np
from huggingface_hub import hf_hub_download
from f5_tts.api import F5TTS

# 1. GPU Kontrolü
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU (CUDA) bulunamadı! Lütfen Çalışma Zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçin.")
device = "cuda"

# Eğer metin.txt dosyası yüklenmişse doğrudan oradan oku
if os.path.exists("metin.txt"):
    with open("metin.txt", "r", encoding="utf-8") as f:
        hedef_metin = f.read()

# 2. Modeli Yükle (Belleğe bir kez alınır)
if 'f5tts' not in globals():
    print(f"⏳ F5-TTS Türkçe modeli GPU belleğine alınıyor ({torch.cuda.get_device_name(0)})...")
    ckpt_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="f5_tts_turkish_1000000.safetensors")
    vocab_path = hf_hub_download(repo_id="marduk-ra/F5-TTS-Turkish", filename="vocab.txt")
    
    f5tts = F5TTS(
        model='F5TTS_Base',
        ckpt_file=ckpt_path,
        vocab_file=vocab_path,
        device=device
    )
    print(f"✅ Model hazır! Çalışma ortamı: {torch.cuda.get_device_name(0)} ({device.upper()})")

# 3. Fonetik İyileştirmeler
fonetik_sozluk = {
    "React": "Riyekt",
    "HTML": "He-Te-Me-Le",
    "JavaScript": "Cavasikript",
    "DOM": "Dom",
    "CSS": "Si-Es-Es",
    "Next.js": "Nekst ceyes",
    "API": "Eypiyay",
    "JSON": "Ceysın",
    "Suspense": "Saspens",
    "fallback": "folbek",
    "transition": "trenzışın",
    "effect": "efekt",
    "render": "rendır",
    "URL": "Yu-Ar-El",
    "AI": "Ey-Ay",
    "LLM": "El-El-Em",
    "TTS": "Ti-Ti-Es",
    "n8n": "En-sekiz-en",
    "WordPress": "Vordpres"
}

islenmis_metin = hedef_metin
for ing, tr in fonetik_sozluk.items():
    islenmis_metin = re.sub(rf'\b{re.escape(ing)}\b', tr, islenmis_metin, flags=re.IGNORECASE)

# Referans ses yolunu al
ref_ses = temiz_referans_yolu if 'temiz_referans_yolu' in globals() else (temiz_ses_yolu if 'temiz_ses_yolu' in globals() else 'temiz_referans.wav')
adim_sayisi = kalite_adim_sayisi if 'kalite_adim_sayisi' in globals() else 32

# 4. Cümle Cümle Hızlı GPU Sentezleme
cumleler = [c.strip() for c in re.split(r'(?<=[.!?])\s+', islenmis_metin.replace('\n', ' ')) if len(c.strip()) > 3]
print(f"🎙️ Seslendirme GPU üzerinde başladı (Toplam {len(cumleler)} cümle, NFE adımı: {adim_sayisi})...\n")

ses_parcalari = []
sr_val = 24000
baslangic = time.time()

for i, cumle in enumerate(cumleler, 1):
    t_cumle = time.time()
    with torch.inference_mode():
        wav, sr, _ = f5tts.infer(
            ref_file=ref_ses,
            ref_text=referans_metni,
            gen_text=cumle,
            nfe_step=adim_sayisi,
            cfg_strength=sadakat_cfg,
            speed=konusma_hizi,
            seed=42,
            remove_silence=False
        )
    sr_val = sr
    ses_parcalari.append(wav)
    # Cümleler arası 250ms doğal insan nefesi payı
    ses_parcalari.append(np.zeros(int(0.25 * sr)))
    print(f"[{i}/{len(cumleler)}] Tamamlandı: \"{cumle[:38]}...\" ({time.time()-t_cumle:.1f} sn)")

# Tek parça halinde birleştir
final_wav = np.concatenate(ses_parcalari)
cikti_dosyasi = "turkce_klonlanmis_ses.wav"
sf.write(cikti_dosyasi, final_wav, sr_val)

toplam_sure = len(final_wav) / sr_val
gecen_zaman = time.time() - baslangic
print(f"\n🎉 Tebrikler! Toplam Ses Süresi: {toplam_sure:.1f} sn | Üretim Süresi: {gecen_zaman:.1f} sn ({gecen_zaman/len(cumleler):.1f} sn/cümle)")


In [ ]:
#@title 📥 5. Adım: Sesi Dinle ve Bilgisayarına İndir
#@markdown Üretilen sesi dinleyin ve otomatik olarak bilgisayarınıza indirin:

from IPython.display import Audio, display
from google.colab import files

display(Audio(cikti_dosyasi, rate=sr_val))
files.download(cikti_dosyasi)
print("✅ Dosya indirme işlemi başlatıldı!")

---
### 🔗 Bağlantılar ve İletişim

Bu proje, açık kaynaklı **F5-TTS** mimarisinin Türkçe ağırlıkları (`marduk-ra/F5-TTS-Turkish`) üzerinde sıfır örnekli (zero-shot) ses klonlama deneyimini herkes için kolaylaştırmak amacıyla geliştirilmiştir.

* 🌐 **Web Sitesi & Blog:** [beydahsaglam.com](https://beydahsaglam.com)
* 💼 **LinkedIn:** [linkedin.com/in/beydah](https://linkedin.com/in/beydah)
* 🐙 **GitHub:** [github.com/beydah](https://github.com/beydah)
* 📄 **F5-TTS Makalesi:** [F5-TTS: A Fairytaler that Fakes Fluent and Faithful Speech with Flow Matching](https://arxiv.org/abs/2410.06885)